# 05 Financial KPI Calculations
**Project:** Retailmart Finance Analytics  
**Phase:** Compute key financial metrics (Gross Revenue, Net Revenue, COGS, Profit Margins).

---

### 1. Setup & Imports
We must add the parent directory to the system path so Python can locate the custom `src` modules.

In [1]:
import sys
import os
import pandas as pd
import numpy as np

# Add root directory to python path
sys.path.append(os.path.abspath(os.path.join('..')))

# Import custom configurations and utilities
from src.config.config import config
from src.database.connection import db_connector
from src.utils.logger import setup_logger
from src.utils.helpers import save_dataframe

from src.utils.helpers import format_large_number

logger = setup_logger(os.path.splitext(os.path.basename('__file__'))[0])
logger.info('Notebook setup and imports loaded successfully.')

[2026-07-19 15:22:26] [INFO] [config.py:16] Loaded environment configuration from: E:\SQL ACCIO JOB\Cross functional dashboard\.env


[2026-07-19 15:22:27] [INFO] [connection.py:42] SQLAlchemy Database Engine initialized successfully.


[2026-07-19 15:22:27] [INFO] [1329643734.py:18] Notebook setup and imports loaded successfully.


### 2. Load Cleaned Data
Placeholder for load cleaned data steps.

In [2]:
import os

# 1. Define paths to our processed clean files
processed_dir = "../data/processed"

# 2. Load the orders table, telling Pandas to parse 'order_date' as a date
orders_df = pd.read_csv(
    os.path.join(processed_dir, "sales_transaction_orders.csv"),
    parse_dates=["order_date"]
)

# 3. Load the general ledger entries, parsing transaction dates
ledger_df = pd.read_csv(
    os.path.join(processed_dir, "finance_accounting_ledger_entries.csv"),
    parse_dates=["transaction_date", "posted_at"]
)

# 4. Load lookup tables
gl_accounts_df = pd.read_csv(os.path.join(processed_dir, "finance_accounting_gl_accounts.csv"))
dealers_df = pd.read_csv(os.path.join(processed_dir, "dealer_dealers.csv"))

# 5. Print shapes to confirm
print(f"Loaded Orders: {orders_df.shape}")
print(f"Loaded Ledger Entries: {ledger_df.shape}")
print(f"Loaded GL Accounts: {gl_accounts_df.shape}")
print(f"Loaded Dealers: {dealers_df.shape}")

Loaded Orders: (7000, 16)
Loaded Ledger Entries: (15000, 10)
Loaded GL Accounts: (80, 6)
Loaded Dealers: (200, 16)


### 3. Compute Monthly Revenue & Costs
Placeholder for compute monthly revenue & costs steps.

In [3]:
# 1. Create a Year-Month column for our orders
orders_df["order_month"] = orders_df["order_date"].dt.to_period("M")

# 2. Calculate Monthly Vehicle Revenue
monthly_vehicle_revenue = orders_df.groupby("order_month")["net_amount"].sum().reset_index()
monthly_vehicle_revenue.rename(columns={"net_amount": "vehicle_revenue"}, inplace=True)

# 3. Join the Ledger Entries with GL Accounts to get account names
merged_ledger = pd.merge(ledger_df, gl_accounts_df, on="account_id")

# 4. Create a Year-Month column for ledger entries
merged_ledger["transaction_month"] = merged_ledger["transaction_date"].dt.to_period("M")

# 5. Extract category codes (first 4 digits of account_code)
merged_ledger["category_code"] = merged_ledger["account_code"].str[:4]

# 6. Separate Expenses (Debit) and Income (Credit)
expenses_df = merged_ledger[merged_ledger["account_type"] == "Expense"]
other_income_df = merged_ledger[merged_ledger["account_type"] == "Income"]

# 7. Aggregate Monthly Expenses by Category
monthly_expenses = expenses_df.groupby(["transaction_month", "category_code"])["debit_amount"].sum().unstack(fill_value=0).reset_index()

# 8. Aggregate Monthly Other Revenues (Spares & Service)
monthly_other_income = other_income_df.groupby(["transaction_month", "category_code"])["credit_amount"].sum().unstack(fill_value=0).reset_index()

# 9. Print sample results
print("Monthly Vehicle Revenue Sample:")
print(monthly_vehicle_revenue.head(3))
print("\nMonthly Expenses Categories Sample:")
print(monthly_expenses.head(3))

Monthly Vehicle Revenue Sample:
  order_month  vehicle_revenue
0     2023-01      81585143.91
1     2023-02      56950148.59
2     2023-03      76857820.03

Monthly Expenses Categories Sample:
category_code transaction_month         5000         5100         5200  \
0                       2023-04   7537430.91   6965070.39   2588839.22   
1                       2023-05  38853936.16  37962864.74  18158302.15   
2                       2023-06  33211904.24  52192753.87  25111609.71   

category_code         5300         5400         5500         5600  
0               7425115.69    428081.98  12026056.88         0.00  
1              11982686.22  20060675.60  42880239.22  20367618.98  
2              16696606.00   6579100.23  80105959.05   4966456.90  


### 4. Calculate Gross and Net Profit Margins
Placeholder for calculate gross and net profit margins steps.

In [4]:
# 1. Standardize month column names before merging
monthly_vehicle_revenue.rename(columns={"order_month": "month"}, inplace=True)
monthly_expenses.rename(columns={"transaction_month": "month"}, inplace=True)
monthly_other_income.rename(columns={"transaction_month": "month"}, inplace=True)

# 2. Merge all monthly sheets together
finance_master = pd.merge(monthly_vehicle_revenue, monthly_other_income, on="month", how="outer")
finance_master = pd.merge(finance_master, monthly_expenses, on="month", how="outer")
finance_master.fillna(0.0, inplace=True)

# 3. Rename category codes to readable business names
rename_map = {
    "4100": "spares_revenue",
    "4200": "service_revenue",
    "5000": "cogs",
    "5100": "manufacturing_costs",
    "5200": "salaries_wages",
    "5300": "marketing_costs",
    "5400": "logistics_costs",
    "5500": "dealer_commissions",
    "5600": "admin_costs"
}
finance_master.rename(columns=rename_map, inplace=True)

# 4. Calculate core financial KPIs
finance_master["total_revenue"] = (
    finance_master["vehicle_revenue"] + 
    finance_master["spares_revenue"] + 
    finance_master["service_revenue"]
)
finance_master["total_expenses"] = (
    finance_master["cogs"] +
    finance_master["manufacturing_costs"] +
    finance_master["salaries_wages"] +
    finance_master["marketing_costs"] +
    finance_master["logistics_costs"] +
    finance_master["dealer_commissions"] +
    finance_master["admin_costs"]
)
finance_master["gross_profit"] = finance_master["total_revenue"] - finance_master["cogs"]
finance_master["net_profit"] = finance_master["total_revenue"] - finance_master["total_expenses"]
finance_master["net_profit_margin"] = (finance_master["net_profit"] / finance_master["total_revenue"]) * 100

# 5. Display master summary sheet
print("--- Master Financial Summary Sheet ---")
display_cols = ["month", "total_revenue", "total_expenses", "gross_profit", "net_profit", "net_profit_margin"]
print(finance_master[display_cols].head(5).to_string(index=False))

--- Master Financial Summary Sheet ---
  month  total_revenue  total_expenses  gross_profit   net_profit  net_profit_margin
2023-01    81585143.91            0.00   81585143.91  81585143.91         100.000000
2023-02    56950148.59            0.00   56950148.59  56950148.59         100.000000
2023-03    76857820.03            0.00   76857820.03  76857820.03         100.000000
2023-04    91400265.70     36970595.07   83862834.79  54429670.63          59.550889
2023-05   400484763.85    190266323.07  361630827.69 210218440.78          52.490996


### 5. Group by Region and Stores
Placeholder for group by region and stores steps.

In [5]:
# 1. Load the commissions table to evaluate dealer payout efficiency
commissions_df = pd.read_csv(os.path.join(processed_dir, "dealer_network_dealer_commissions.csv"))

# 2. Aggregate sales per dealer from the orders table
dealer_sales = orders_df.groupby("dealer_id").agg(
    total_orders=("order_id", "count"),
    total_revenue=("net_amount", "sum"),
    avg_order_value=("net_amount", "mean")
).reset_index()

# 3. Aggregate commissions paid per dealer
dealer_commissions = commissions_df.groupby("dealer_id")["total_commission"].sum().reset_index()

# 4. Merge sales summaries with dealer metadata and commissions
dealer_summary = pd.merge(dealer_sales, dealers_df[["dealer_id", "dealer_name", "status"]], on="dealer_id", how="left")
dealer_summary = pd.merge(dealer_summary, dealer_commissions, on="dealer_id", how="left")
dealer_summary["total_commission"] = dealer_summary["total_commission"].fillna(0.0)

# 5. Calculate payout efficiency ratio
dealer_summary["payout_ratio_pct"] = (dealer_summary["total_commission"] / dealer_summary["total_revenue"]) * 100

# 6. Sort and print top performers
dealer_summary = dealer_summary.sort_values(by="total_revenue", ascending=False)
print("--- Top 5 Performing Showrooms ---")
print(dealer_summary[["dealer_name", "total_orders", "total_revenue", "total_commission", "payout_ratio_pct"]].head(5).to_string(index=False))

--- Top 5 Performing Showrooms ---
            dealer_name  total_orders  total_revenue  total_commission  payout_ratio_pct
    Dada Motors Dhanbad            47    94050873.16        31049213.0         33.013211
  Kar Motors Chandrapur            50    92021301.79        22320709.0         24.256024
  Ahuja Motors Suryapet            45    89153735.34        13713162.0         15.381478
Chaudhry Motors Bikaner            49    86400299.95        19440004.0         22.499927
   Nagi Motors Durgapur            48    85810655.72        24923385.0         29.044627


### 6. Export Finance Aggregates
Placeholder for export finance aggregates steps.

In [6]:
# 1. Define filenames for our aggregate tables
summary_file = os.path.join(processed_dir, "monthly_finance_summary.csv")
dealer_file = os.path.join(processed_dir, "dealer_performance_summary.csv")

# 2. Export the monthly master financial sheet to CSV (excluding index)
success_summary = save_dataframe(finance_master, summary_file, index=False)

# 3. Export the showroom dealer sales rankings
success_dealer = save_dataframe(dealer_summary, dealer_file, index=False)

# 4. Print success statements
if success_summary:
    print(f"Successfully exported Monthly Finance Summary to: {summary_file} (Shape: {finance_master.shape})")
if success_dealer:
    print(f"Successfully exported Dealer Performance Summary to: {dealer_file} (Shape: {dealer_summary.shape})")

[2026-07-19 15:22:27] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (48, 17) to: ../data/processed\monthly_finance_summary.csv


[2026-07-19 15:22:27] [INFO] [helpers.py:59] Successfully saved DataFrame of shape (200, 8) to: ../data/processed\dealer_performance_summary.csv


Successfully exported Monthly Finance Summary to: ../data/processed\monthly_finance_summary.csv (Shape: (48, 17))
Successfully exported Dealer Performance Summary to: ../data/processed\dealer_performance_summary.csv (Shape: (200, 8))
